# 🚀 PromptFlow CLI & Python API Orchestration Tutorial

Welcome to the **PromptFlow Standalone Execution Tutorial**! This Jupyter Notebook demonstrates how to orchestrate PromptFlow capabilities programmatically using:
1. **The Python Programmatic API** (`run_flow_from_file`)
2. **The Standalone CLI Interface** (`python cli.py run`)

### 🌟 Capabilities Covered:
- 📄 **File-Based Execution**: Run `.md` prompt files and `.flow.json` workflow definitions without requiring database persistence.
- 🐍 **Python API Integration**: Programmatic workflow execution directly inside Python scripts and Jupyter Notebooks.
- ⚙️ **Model & Provider Overrides**: Dynamically override AI models (e.g., `gpt-4o`, `gpt-4o-mini`) and providers.
- 💾 **Input & Output Persistence**: Read inputs from JSON files and save structured results to disk.

## 1. Setup Environment & Path Resolution
We begin by adding the `backend` directory to `sys.path` so we can import PromptFlow's file runner modules.

In [1]:
import sys
import json
import subprocess
from pathlib import Path

# Resolve backend directory path
notebook_dir = Path(".").resolve()
backend_dir = (notebook_dir.parent / "backend").resolve() if notebook_dir.name == "notebooks" else (notebook_dir / "backend").resolve()

if str(backend_dir) not in sys.path:
    sys.path.insert(0, str(backend_dir))

print(f"Backend directory configured: {backend_dir}")

from file_runner import run_flow_from_file
print("Successfully imported PromptFlow 'run_flow_from_file' API!")

Backend directory configured: C:\Projects\promptflow\backend
Successfully imported PromptFlow 'run_flow_from_file' API!


## 2. Programmatic Python API Execution
You can execute any Markdown prompt file (`.md`) or Flow file (`.flow.json`) programmatically using `run_flow_from_file`.

Here we execute `backend/prompts/test_prompt.md` with input variables passed as a Python `dict`.

In [2]:
# Define sample input values
inputs = {
    "business_name": "Acme Cloud Analytics",
    "business_content": "Acme Cloud Analytics provides real-time infrastructure metrics and AI-driven cost optimization for Kubernetes clusters."
}

prompt_file = backend_dir / "prompts" / "test_prompt.md"

print(f"Executing Markdown Prompt: {prompt_file.name}")
result = run_flow_from_file(
    flow=prompt_file,
    input_values=inputs
)

print("\n--- Execution Result ---")
print(f"Flow ID: {result['flow_id']}")
print(f"Status : {result['status']}")
print("\nGenerated Output:")
print(result["output"])

Executing Markdown Prompt: test_prompt.md

--- Execution Result ---
Flow ID: test_prompt
Status : completed

Generated Output:
**Business Summary: Acme Cloud Analytics**

Acme Cloud Analytics specializes in delivering real-time infrastructure metrics and AI-driven cost optimization solutions specifically for Kubernetes clusters. By leveraging advanced analytics and machine learning, the company empowers organizations to enhance their operational efficiency, reduce costs, and optimize resource allocation in cloud environments. With a focus on Kubernetes, a leading container orchestration platform, Acme Cloud Analytics addresses the growing demand for tools that facilitate better management and performance of cloud-native applications. The business positions itself as a key player in the cloud analytics market, catering to enterprises seeking to maximize their cloud infrastructure investments while ensuring scalability and reliability.


## 3. Dynamic Model Overrides & Provider Selection
PromptFlow supports a 3-tier model resolution hierarchy. You can pass a `model_override` dictionary to specify the target model and provider at runtime.

In [3]:
# Model override configuration
custom_model = {
    "provider": "openai",
    "name": "gpt-4o-mini",
    "temperature": 0.5
}

output_file = backend_dir / "outputs" / "notebook_model_override_result.json"

result = run_flow_from_file(
    flow=prompt_file,
    input_values=inputs,
    model_override=custom_model,
    output_path=output_file
)

print(f"Saved output to: {output_file}")
print(f"Output status: {result['status']}")

Saved output to: C:\Projects\promptflow\backend\outputs\notebook_model_override_result.json
Output status: completed


## 4. Multi-Step Markdown Workflow Orchestration
Now let's execute a multi-step Markdown flow (`backend/prompts/business_marketing.md`). This prompt file defines multiple sequential steps (`business_content`, `business_value_themes`, `visual_concepts`, `marketing_caption`).

In [ ]:
marketing_prompt_file = backend_dir / "prompts" / "business_marketing.md"
marketing_inputs = {
    "business_name": "HyperDrive Logistics",
    "business_content": "Autonomous electric drone delivery fleet delivering emergency medical supplies within 15 minutes."
}

marketing_result = run_flow_from_file(
    flow=marketing_prompt_file,
    input_values=marketing_inputs
)

print(f"Completed {len(marketing_result['steps'])} steps.")
for i, step in enumerate(marketing_result['steps'], 1):
    print(f"\nStep {i} ({step.get('step_id', 'step')}): {step.get('status')}")
    res_text = str(step.get('result', ''))[:150] + "..."
    print(f"Result Preview: {res_text}")

## 5. CLI Execution via Subprocess (`cli.py run`)
PromptFlow includes a standalone CLI (`backend/cli.py`). You can invoke it from shell commands or Python subprocesses.

### Creating a JSON Input File:

In [4]:
# Write a sample JSON input file
input_json_path = backend_dir / "sample_input.json"
sample_data = {
    "business_name": "Solaris Energy Systems",
    "business_content": "Solaris builds smart solar micro-grids with automated battery storage balancing for commercial buildings."
}
input_json_path.write_text(json.dumps(sample_data, indent=2), encoding="utf-8")
print(f"Created sample input JSON at: {input_json_path}")

Created sample input JSON at: C:\Projects\promptflow\backend\sample_input.json


### Executing CLI Command:
Run `python cli.py run prompts/test_prompt.md --input sample_input.json --output outputs/cli_output.json --model gpt-4o-mini --provider openai`

In [5]:
cli_path = backend_dir / "cli.py"
output_json_path = backend_dir / "outputs" / "cli_output.json"

cmd = [
    sys.executable,
    str(cli_path),
    "run",
    str(prompt_file),
    "--input", str(input_json_path),
    "--output", str(output_json_path),
    "--model", "gpt-4o-mini",
    "--provider", "openai"
]

print("Running CLI Command:", " ".join(cmd))
completed_process = subprocess.run(cmd, capture_output=True, text=True, cwd=str(backend_dir))

print("\n--- CLI STDOUT ---")
print(completed_process.stdout)
if completed_process.stderr:
    print("--- CLI STDERR ---")
    print(completed_process.stderr)

Running CLI Command: c:\Projects\promptflow\.venv\Scripts\python.exe C:\Projects\promptflow\backend\cli.py run C:\Projects\promptflow\backend\prompts\test_prompt.md --input C:\Projects\promptflow\backend\sample_input.json --output C:\Projects\promptflow\backend\outputs\cli_output.json --model gpt-4o-mini --provider openai

--- CLI STDOUT ---
Flow 'test_prompt' executed successfully!
Output saved to: C:\Projects\promptflow\backend\outputs\cli_output.json



## 6. Inspecting the Output JSON Artifact
Let's read and parse the generated `cli_output.json` file to verify the structured output.

In [6]:
if output_json_path.exists():
    saved_payload = json.loads(output_json_path.read_text(encoding="utf-8"))
    print("Keys in output JSON payload:", list(saved_payload.keys()))
    print("\nFlow ID:", saved_payload.get("flow_id"))
    print("Status :", saved_payload.get("status"))
    print("\nFinal Output Content:\n", saved_payload.get("output"))
else:
    print(f"File not found: {output_json_path}")

Keys in output JSON payload: ['flow_id', 'flow_name', 'status', 'output', 'context', 'steps']

Flow ID: test_prompt
Status : completed

Final Output Content:
 **Business Summary: Solaris Energy Systems**

Solaris Energy Systems specializes in the development of advanced solar micro-grids equipped with automated battery storage balancing solutions tailored for commercial buildings. By integrating smart technology with renewable energy sources, the company aims to enhance energy efficiency, reduce operational costs, and promote sustainability for businesses. Solaris focuses on delivering reliable, scalable, and intelligent energy solutions that empower commercial properties to harness solar power effectively while ensuring optimal energy management through automated systems. This innovative approach positions Solaris as a key player in the growing market for clean energy infrastructure and smart building technologies.


## 7. Summary & Best Practices

- 💡 **Standalone Execution**: You can execute flows without starting the Flask web server or PostgreSQL database.
- 🐍 **Jupyter Notebook Integration**: Import `run_flow_from_file` from `file_runner` for instant notebook experimentation.
- ⚡ **CLI Automation**: Use `python cli.py run` in CI/CD pipelines, cron jobs, or batch scripts with `--input` and `--output` flags.